# Exercise 04 - Implementation of Automatic Differentiation
In this notebook, you will implement automatic differentiation for scalars.

Based on https://github.com/kennysong/minigrad.
You can check your implementation with their code (https://github.com/kennysong/minigrad/blob/master/tests.ipynb), which requires installing some additional packages.
You can also compare your results with PyTorch's autograd.
There is also a simple test case at the end of the notebook.
We strongly recommend looking at their code only after you have completed the assignment!

Implement the methods marked with TODO.

In [ ]:
from __future__ import annotations

"""
This class keeps track of the computation graph and implements backprop.
Each instance has an attribute `input_nodes`, which keeps track of the inputs. 
Recursively traversing these lists yields the full computation graph.
The `value` attribute stores the actual scalar value.
The `grad` attribute stores ∂output/∂self, where output is the final scalar from which the backward pass starts. 
Initially it is zero, and it will only be changed during backpropagation.
The `grad_wrt` ("gradient with respect to") attribute is a dictionary that stores the local derivatives ∂self/∂input_node. 
These values must be filled in when the operation (+, -, ...) is executed, i.e., during the forward pass.
This class overloads Python's arithmetic operators. You must implement the missing ones; an example is in the __add__ method.
Finally, the `backward` method implements the backpropagation.
"""


class Scalar:
    def __init__(
        self,
        value: float,
        input_nodes: list[Scalar] | None = None,
        creation_op: str | None = None,
    ):
        self.value = value
        self.input_nodes = (
            input_nodes if input_nodes else []
        )  # Scalars this node depends on (nodes closer to input)
        self.creation_op = creation_op  # A string for visualization of operations, e.g., if this object was created by adding two Scalars, then creation_op = "+"
        self.grad = 0  # This stores the ∂output/∂self value
        self.grad_wrt = {}  # This dict stores all ∂self/∂input_node values
        # (only populated if self has input_nodes, i.e. self was created by an arithmetic op)

    def __repr__(self) -> str:
        return f"Scalar(value={self.value:.2f}, grad={self.grad:.2f})"

    # Called by: self + other
    # Example method. We first check whether `other` is a `Scalar`; if not, we wrap it in one.
    # Then we instantiate a new Scalar object, which will be the return value of this operation.
    # For this new object, we define the partial derivatives of the output with respect to (wrt) its input nodes.
    # This information is needed for backpropagation.
    # Implementing this method defines how the standard "+" operator works on objects of type Scalar.
    def __add__(self, other) -> Scalar:
        if not isinstance(other, Scalar):
            other = Scalar(other)

        output = Scalar(self.value + other.value, [self, other], creation_op="+")
        output.grad_wrt[self] = 1
        output.grad_wrt[other] = 1
        return output

    # Called by: other + self
    def __radd__(self, other) -> Scalar:
        # TODO
        pass

    # Called by: self - other
    def __sub__(self, other) -> Scalar:
        # TODO
        pass

    # Called by: other - self
    def __rsub__(self, other) -> Scalar:
        # TODO
        pass

    # Called by: self * other
    def __mul__(self, other) -> Scalar:
        # TODO
        pass

    # Called by: other * self
    def __rmul__(self, other) -> Scalar:
        # TODO
        pass

    # Called by: self / other
    def __truediv__(self, other) -> Scalar:
        # TODO
        pass

    # Called by: other / self
    def __rtruediv__(self, other) -> Scalar:
        # TODO
        pass

    # Called by: self**other
    def __pow__(self, other) -> Scalar:
        assert isinstance(other, (int, float)), (
            "minigrad does not support a Scalar in the exponent"
        )

        # TODO

    # Called by: -self
    def __neg__(self) -> Scalar:
        # TODO
        pass

    # This method does the backpropagation and is called on the final output node (usually the loss)
    def backward(self):
        """Compute ∂self/∂node, i.e. ∂output/∂node, for each node in self's dependency graph."""

        def _topological_order() -> list[Scalar]:
            r"""Returns a topological ordering of self and all nodes it depends on. As an example:
            For the graph x --> t1 --> t4
                             \       /
                              \     /
                          y --> t2 /
            [x, t1, y, t2, t4] is a topological ordering. (There are generally multiple correct orderings.)
            If you get stuck, https://en.wikipedia.org/wiki/Topological_sorting may help.
            """
            # TODO (Hint: you can write a recursive solution)

        def _compute_grad_of_parents(node: Scalar):
            """Given a node, compute its input_nodes' gradients: ∂output/∂input_node = ∂output/∂node * ∂node/∂input_node."""
            # (Hint: A node may be the input of several other nodes;
            # each of them contributes to its gradient, so accumulate with += instead of assigning.)
            # TODO

        # ∂output/∂output = 1; this bootstraps the backpropagation
        self.grad = 1

        # Now traverse through the graph in order from output to input, and compute gradients!
        ordered = reversed(_topological_order())
        for node in ordered:
            _compute_grad_of_parents(node)

A simple test case.

In [ ]:
y = Scalar(5)
y_hat = Scalar(3)
output = (y_hat - y) ** 2  # Squared loss

# manually: ∂output/∂y_hat = 2 * (y_hat - y)
manual_grad = (2 * (y_hat - y)).value
# automatically
output.backward()

print(f"y = {y.value}, y_hat = {y_hat.value}")
print(f"output = (y_hat - y)^2 = {output.value}")
print(f"∂output/∂y_hat (manual)    = {manual_grad}")
print(f"∂output/∂y_hat (autograd)  = {y_hat.grad}")
print(
    f"∂output/∂y     (autograd)  = {y.grad}"
)  # manually: ∂output/∂y = 2 * (y_hat - y) * (-1); thus, sign flip!

assert y_hat.grad == manual_grad, f"{y_hat.grad=} != {manual_grad=}"
# print("Gradients match!")